In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# SHARED PROTEIN INTERACTION

# BioGrid Data
Downloaded from: https://downloads.thebiogrid.org/BioGRID/Release-Archive/BIOGRID-4.4.241/

In [5]:
def process_biogrid_ppi_data(data):
    # Filter interactions down to physical interactions between two human proteins
    hsapien = 9606
    biogrid_ppi = data[(data['Organism ID Interactor A']==hsapien) & (data['Organism ID Interactor B']==hsapien) &
                       (data['Experimental System Type']=='physical')]
    # Filter to columns of interest + rename
    biogrid_ppi = biogrid_ppi[['Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'Experimental System']]
    biogrid_ppi = biogrid_ppi.rename(columns={'Entrez Gene Interactor A':'A1_entrez', 'Entrez Gene Interactor B':'A2_entrez',
                                              'Experimental System':'experimental_system'})
    biogrid_ppi = biogrid_ppi.drop_duplicates().reset_index(drop=True).astype({'A1_entrez':'int', 'A2_entrez':'int'})
    
    # Filter out self interactions
    biogrid_ppi = biogrid_ppi[biogrid_ppi.A1_entrez != biogrid_ppi.A2_entrez].reset_index(drop=True)
    print('N interactions:', biogrid_ppi.shape[0])
    
    # Sort interacting genes - to find unique interaction pairs
    biogrid_unique = pd.DataFrame(np.sort(biogrid_ppi[['A1_entrez','A2_entrez']], axis=1), 
                                  columns=['A1_entrez', 'A2_entrez']).drop_duplicates()
    assert(biogrid_unique.shape[0] <= biogrid_ppi.shape[0])

    print('N interactions after sorting for unique pairs:', biogrid_unique.shape[0])
    print('N genes in interaction map:', pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).nunique())
    
    return biogrid_unique

def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df
def run_pairwise_ppi_features_for_pair(biomarker: str, target1: str, biogrid_unique: pd.DataFrame):
    """
    Compute PPI overlap features (biomarker–query and target1–query)
    for one biomarker–target pair.

    Parameters
    ----------
    biomarker : str
        Biomarker gene symbol (e.g. "BRCA1").
    target1 : str
        Target1 gene symbol (e.g. "POLQ").
    biogrid_unique : pd.DataFrame
        Unique BioGRID interactions (output from process_biogrid_ppi_data).

    Returns
    -------
    biomarker_query_fet : pd.DataFrame
    target1_query_fet : pd.DataFrame
    """
    # Define file naming
    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    # Load dataset
    main_file_csv = pd.read_csv(main_file, low_memory=False)

    # --- Biomarker vs Query ---
    biomarker_query = main_file_csv[['entrez_id_biomarker', 'entrez_id_query']].rename(
        columns={'entrez_id_biomarker': 'A1_entrez', 'entrez_id_query': 'A2_entrez'}
    )
    biomarker_query_fet = compute_ppi_summary_for_pairs(biogrid_unique, biomarker_query)
    biomarker_query_fet.to_csv(f'feature_output/fet_ppi_overlap_biomarker_query_{SL_pair}.csv', index=False)

    # --- Target1 vs Query ---
    target1_query = main_file_csv[['entrez_id_target1', 'entrez_id_query']].rename(
        columns={'entrez_id_target1': 'A1_entrez', 'entrez_id_query': 'A2_entrez'}
    )
    target1_query_fet = compute_ppi_summary_for_pairs(biogrid_unique, target1_query)
    target1_query_fet.to_csv(f'feature_output/fet_ppi_overlap_target1_query_{SL_pair}.csv', index=False)

    print(f"✅ PPI features computed for {SL_pair}")
    return biomarker_query_fet, target1_query_fet

# Load BioGRID only once (done outside the loop)
biogrid_raw = pd.read_csv(
    "/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/BIOGRID-ALL-4.4.241.tab3.txt",
    sep="\t", low_memory=False
)
biogrid_unique = process_biogrid_ppi_data(biogrid_raw)

# Load pairs from Excel
pairs_df = pd.read_excel("biomarker_target_genes.xlsx")  # Expected cols: Biomarker, Target

# Loop through pairs
for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    run_pairwise_ppi_features_for_pair(biomarker, target1, biogrid_unique)


N interactions: 993165
N interactions after sorting for unique pairs: 862452
N genes in interaction map: 19995
N. gene pairs that interact: 257
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
N. gene pairs that interact: 92
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
✅ PPI features computed for MSH6_WRN
N. gene pairs that interact: 713
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
N. gene pairs that interact: 92
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
✅ PPI features computed for CDH1_ROS1
N. gene pairs that interact: 303
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
N. gene pairs that interact: 197
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19995
✅ PPI features computed for ATM_ATR
N. gene pair

# STRING data

In [6]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]
string = string[string['combined_score'] >= 400].sort_values(by='combined_score', ascending=False)

string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)

# === HGNC lookup
gene_lookup = {}
def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)
    entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
    syms = []
    s = normalize_symbol(entry.get('symbol'))
    if s: syms.append(s)
    if pd.notnull(entry.get('prev_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
    if pd.notnull(entry.get('alias_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
    for sym in syms:
        if sym and sym not in gene_lookup:
            gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

string['hgnc_id_1'], string['ensembl_gene_id_1'], string['entrez_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'], string['entrez_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))

string_unique = (
    string[["entrez_id_1", "entrez_id_2"]]
    .rename(columns={"entrez_id_1": "A1_entrez", "entrez_id_2": "A2_entrez"})
    .drop_duplicates()
    .reset_index(drop=True)
)

# Convert safely, making "NA" into NaN first
string_unique["A1_entrez"] = pd.to_numeric(string_unique["A1_entrez"], errors="coerce").astype("Int64")
string_unique["A2_entrez"] = pd.to_numeric(string_unique["A2_entrez"], errors="coerce").astype("Int64")

print('N interactions:', string_unique.shape[0])

# Drop rows with missing values before sorting
string_unique = string_unique.dropna(subset=["A1_entrez", "A2_entrez"])

# Sort interacting genes - to find unique interaction pairs
string_unique = (
    pd.DataFrame(np.sort(string_unique[['A1_entrez','A2_entrez']], axis=1), 
                 columns=['A1_entrez', 'A2_entrez'])
    .drop_duplicates()
)
string_unique = string_unique[string_unique["A1_entrez"] != string_unique["A2_entrez"]]

print('N interactions after sorting for unique pairs:', string_unique.shape[0])
print('N genes in interaction map:', pd.concat([string_unique.A1_entrez, string_unique.A2_entrez]).nunique())

## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df
import pandas as pd

def run_pairwise_string_ppi_features(main_csv_path: str, string_unique: pd.DataFrame, output_prefix: str):
    """
    Compute STRING-based PPI overlap features (biomarker–query and target1–query)
    for a given dataset CSV file.

    Parameters
    ----------
    main_csv_path : str
        Path to the dataset (e.g. 'datasets/PredictingSLResistanceFeatures_BRCA1_POLQ_main.csv').
    string_unique : pd.DataFrame
        Unique STRING interactions (already preprocessed).
    output_prefix : str
        Prefix for output files (e.g. 'BRCA1_POLQ').

    Returns
    -------
    biomarker_query_fet : pd.DataFrame
    target1_query_fet : pd.DataFrame
    """
    # Load dataset
    main_file_csv = pd.read_csv(main_csv_path, low_memory=False)

    # --- Biomarker vs Query ---
    biomarker_query = main_file_csv[['entrez_id_biomarker', 'entrez_id_query']].rename(
        columns={'entrez_id_biomarker': 'A1_entrez', 'entrez_id_query': 'A2_entrez'}
    )
    biomarker_query_fet = compute_ppi_summary_for_pairs(string_unique, biomarker_query)
    biomarker_query_fet.to_csv(f'feature_output/fet_ppi_overlap_biomarker_query_string_{output_prefix}.csv',
                               index=False)

    # --- Target1 vs Query ---
    target1_query = main_file_csv[['entrez_id_target1', 'entrez_id_query']].rename(
        columns={'entrez_id_target1': 'A1_entrez', 'entrez_id_query': 'A2_entrez'}
    )
    target1_query_fet = compute_ppi_summary_for_pairs(string_unique, target1_query)
    target1_query_fet.to_csv(f'feature_output/fet_ppi_overlap_target1_query_string_{output_prefix}.csv',
                             index=False)

    print(f"✅ STRING PPI features computed for {output_prefix}")
    return biomarker_query_fet, target1_query_fet

# === Config ===
excel_file = "biomarker_target_genes.xlsx"   # Excel file with Biomarker + Target columns

# Read biomarker–target pairs
pairs_df = pd.read_excel(excel_file)  
# Expected columns: "Biomarker", "Target"

# Loop through pairs
for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"
    main_file = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"

    run_pairwise_string_ppi_features(
        main_csv_path=main_file,
        string_unique=string_unique,   # already preprocessed earlier
        output_prefix=SL_pair
    )



/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_78605/123532328.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


N interactions: 1840450
N interactions after sorting for unique pairs: 915312
N genes in interaction map: 19004
N. gene pairs that interact: 324
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19004
N. gene pairs that interact: 326
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19004
✅ STRING PPI features computed for MSH6_WRN
N. gene pairs that interact: 872
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19004
N. gene pairs that interact: 95
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19004
✅ STRING PPI features computed for CDH1_ROS1
N. gene pairs that interact: 761
N paralog pairs w/ 1+ interactor (A1 and/or A2): 19271
N genes involded in interactions: 19004
N. gene pairs that interact: 0
N paralog pairs w/ 1+ interactor (A1 and/or A2): 18755
N genes involded in interactions: 19004
✅ STRING PPI features computed for